# DataLens on Colab: train with GRPO, evaluate against frontier models

This notebook runs the full experiment on one GPU:

1. download BIRD,
2. score the zero-shot base model,
3. fine-tune it with GRPO (execution reward),
4. score the fine-tuned model,
5. score the frontier models through the API,
6. build the comparison report.

**Runtime:** an A100 for the 3B configs (Runtime > Change runtime type). The 1.5B configs fit an L4.
Everything is plain `datalens` CLI calls, so the same commands work on any GPU machine.

**If the runtime disconnects,** rerun the Setup and Data cells, then the cell you were on. Training
continues from its last checkpoint on Drive, and evaluation skips questions it already answered.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## Setup
Runs, checkpoints and reports are written to Google Drive so a disconnected runtime doesn't lose them.
BIRD itself stays on the runtime's local disk: train is over 30 GB unpacked, more than a free Drive holds.
The outputs need about 16 GB of Drive space, mostly the two merged models.

The install can end with pip reporting "dependency conflicts" for packages Colab preinstalls (such as
RAPIDS cuDF/cuML or google-adk). The notebook doesn't use them, so the warning is harmless. The check
after the install confirms the training stack works.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

WORKDIR = "/content/drive/MyDrive/datalens"   # persistent outputs
BRANCH = "main"                                # or a pull request's branch, to try it before merging

In [ ]:
%cd /content
!git clone -q -b {BRANCH} https://github.com/RohanSada/DataLens.git || (cd DataLens && git pull -q)
%cd /content/DataLens
# pip keeps Colab's torch and picks the vLLM built for it (0.26 on torch 2.11), which was
# tested with transformers 5.13.1.
!pip install -q -e ".[train,eval,vllm]" "transformers==5.13.1"
# Colab's torchao is too old for peft, which then refuses to train. Nothing here uses torchao.
!pip uninstall -q -y torchao
!mkdir -p {WORKDIR}/runs {WORKDIR}/checkpoints {WORKDIR}/reports
!ln -sfn {WORKDIR}/runs runs && ln -sfn {WORKDIR}/checkpoints checkpoints && ln -sfn {WORKDIR}/reports reports

In [ ]:
# Should print GPU: True
!python -c "import torch, trl, vllm; print('GPU:', torch.cuda.is_available(), '| torch', torch.__version__, '| trl', trl.__version__, '| vllm', vllm.__version__)"

## 1. Data
Downloads into `data/bird/` on the runtime's own disk, so a new runtime downloads it again. A split that is
already in place is skipped. Train is tens of GB unpacked, so this cell takes a while.

In [ ]:
!datalens data download --split dev --split train

## 2. Zero-shot baseline
`eval run` generates (GPU) and then scores (CPU). Generation resumes if interrupted.

In [ ]:
!datalens eval run configs/eval/base-qwen2.5-coder-3b.yaml

## 3. GRPO
Optional but recommended: run the base model on train with 8 samples and drop questions it always (or never)
solves, since those give GRPO no learning signal.

In [ ]:
# Optional difficulty filter
# !datalens eval run configs/eval/base-qwen2.5-coder-3b.yaml --benchmark bird-train --run-name base-3b-train --set decoding.samples=8 --set decoding.greedy=false
# !datalens data exclude-solved runs/base-3b-train/scored.jsonl --out runs/base-3b-train/exclude_ids.txt

In [ ]:
!datalens train grpo configs/train/grpo-qwen2.5-coder-3b.yaml
# rerunning this cell after a disconnect resumes from the last checkpoint
# with the filter: add --set data.exclude_ids_file=runs/base-3b-train/exclude_ids.txt
# out of GPU memory: add --set trainer.per_device_train_batch_size=4 --set trainer.gradient_accumulation_steps=16

In [ ]:
!datalens train plot checkpoints/grpo-qwen2.5-coder-3b/trainer_state.json --out reports/bird-dev/figures

## 4. Score the fine-tuned model

In [ ]:
!datalens eval run configs/eval/grpo-qwen2.5-coder-3b.yaml

## 5. SFT baseline (optional)
Plain fine-tuning on the same data, to show what GRPO adds. The report includes it when it exists.

In [ ]:
!datalens train sft configs/train/sft-qwen2.5-coder-3b.yaml
!datalens eval run configs/eval/sft-qwen2.5-coder-3b.yaml

## 6. Frontier models
Add `ANTHROPIC_API_KEY` under Colab's *Secrets* (key icon) and turn on its notebook access. Scoring all 1,534 questions costs money:
check the prices in the config files first.

In [ ]:
import os
from google.colab import userdata
os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
!datalens eval run configs/eval/frontier-claude-sonnet-5-5.yaml
!datalens eval run configs/eval/frontier-claude-opus-5-5.yaml

## 7. Report

In [ ]:
import os
from IPython.display import Markdown, display

candidates = ["base-qwen2.5-coder-3b", "grpo-qwen2.5-coder-3b", "sft-qwen2.5-coder-3b",
              "frontier-claude-sonnet-5-5", "frontier-claude-opus-5-5"]
# Only runs that finished scoring: a Claude run that failed (e.g. no API key) still leaves a folder.
run_dirs = " ".join(f"runs/{name}" for name in candidates if os.path.isfile(f"runs/{name}/scored.jsonl"))
!datalens eval report {run_dirs} --out reports/bird-dev
display(Markdown(open("reports/bird-dev/results.md").read()))

## 8. Publish the model (optional)
Set `export.hub_repo` in the training config, or push the merged folder yourself.

In [ ]:
# from huggingface_hub import notebook_login; notebook_login()
# !hf upload your-user/datalens-qwen2.5-coder-3b-grpo checkpoints/grpo-qwen2.5-coder-3b/merged